# Exercise 03 — BookingGuard: Protecting Hotel Revenue

> **Graded individual exercise.** Build a time-aware cancellation model and a defensible deposit policy. The model estimates risk; the threshold and published costs define the operating policy.

**Class boundary:** In Week 7, complete Sections 0–5 through the first fair model comparison and notebook debrief. Stop at the marked class hard stop. Sections 6–9 are independent work and must contain your own experiments, threshold choice, sensitivity analysis, group analysis, and recommendation.

Read the [exercise brief](../README.md), [data dictionary](../data/data_dictionary.md), and [rubric](../rubric.md) before starting.

## Before you run anything

**Prediction question:** What is the probability that a booking will cancel, using only information available at the deposit decision point?

**Decision question:** Which bookings should require a deposit under uncertain lost-room value and guest-friction costs?

The later-time test set remains untouched until your model and policy decisions are recorded.

In [ ]:
from pathlib import Path
import platform, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import log_loss, roc_auc_score, average_precision_score, brier_score_loss
SEED = 42
np.random.seed(SEED)
warnings.filterwarnings('ignore', category=FutureWarning)
print({'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit_learn': sklearn_version, 'seed': SEED})

## 0. Reproducible data setup

The loader uses a local repository checkout when available. In Colab, it reads the same public repository files. Record the resolved source and row counts.

In [ ]:
LOCAL_CANDIDATES = [Path('../data'), Path('data'), Path('/content/modern-ai-ml-exercises/exercise-03-booking-guard/data')]
LOCAL_DATA = next((p for p in LOCAL_CANDIDATES if (p/'train.csv').exists()), None)
RAW = 'https://raw.githubusercontent.com/lathrahul/modern-ai-ml-exercises/main/exercise-03-booking-guard/data'
def read_course_csv(name):
    source = LOCAL_DATA/name if LOCAL_DATA else f'{RAW}/{name}'
    return pd.read_csv(source), str(source)
train, train_source = read_course_csv('train.csv')
test, test_source = read_course_csv('test.csv')
costs, costs_source = read_course_csv('policy_costs.csv')
print({'train_source': train_source, 'test_source': test_source, 'cost_source': costs_source, 'train_shape': train.shape, 'test_shape': test.shape})
display(train.head(3), costs)

## 1. Prediction moment and leakage audit

Write the operational moment in one sentence. For every feature, ask whether it would be known then. Do not use `canceled`, reconstructed source outcomes, reservation status, or post-arrival information.

- Prediction moment: **[write here]**
- Fields excluded for leakage or timing: **[write here]**
- Remaining uncertainty: **[write here]**

In [ ]:
TARGET, ID = 'canceled', 'booking_id'
assert TARGET in train and TARGET not in test
assert train[ID].is_unique and test[ID].is_unique
display(pd.DataFrame({'dtype': train.dtypes.astype(str), 'missing_rate': train.isna().mean()}).sort_values('missing_rate', ascending=False).head(12))
print('Target prevalence:', train[TARGET].mean().round(4))

## 2. Temporal validation and baselines

Use earlier arrivals for training and the most recent pre-July-2017 arrivals for validation. The code declares the boundary and prints dates, row counts, and class rates. Do not tune on `test.csv`.

In [ ]:
def arrival_key(df):
    return df['arrival_year'].astype(int)*10000 + df['arrival_week'].astype(int)*10 + df['arrival_day'].astype(int)
ordered = train.assign(_arrival_key=arrival_key(train)).sort_values('_arrival_key').reset_index(drop=True)
VALIDATION_FRACTION = 0.20
cut = int(len(ordered)*(1-VALIDATION_FRACTION))
train_period, valid_period = ordered.iloc[:cut].copy(), ordered.iloc[cut:].copy()
assert train_period['_arrival_key'].max() <= valid_period['_arrival_key'].min()
print({'train_rows':len(train_period),'validation_rows':len(valid_period),'train_key_max':int(train_period['_arrival_key'].max()),'validation_key_min':int(valid_period['_arrival_key'].min()),'train_prevalence':train_period[TARGET].mean(),'validation_prevalence':valid_period[TARGET].mean()})

In [ ]:
FEATURES = [c for c in train.columns if c not in {ID,TARGET}]
numeric = train[FEATURES].select_dtypes(include=np.number).columns.tolist()
categorical = [c for c in FEATURES if c not in numeric]
numeric_pipe = Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
categorical_pipe = Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num',numeric_pipe,numeric),('cat',categorical_pipe,categorical)])
X_train,y_train = train_period[FEATURES],train_period[TARGET]
X_valid,y_valid = valid_period[FEATURES],valid_period[TARGET]
print({'numeric_features':len(numeric),'categorical_features':len(categorical)})

## 3. Logistic probability baseline

This establishes the evidence a more flexible learner must improve. All preprocessing is learned inside the pipeline.

In [ ]:
def probability_metrics(y,p):
    return {'log_loss':log_loss(y,p),'pr_auc':average_precision_score(y,p),'roc_auc':roc_auc_score(y,p),'brier':brier_score_loss(y,p)}
experiments,models,probabilities = [],{},{}
def fit_and_record(name,model,configuration,hypothesis):
    started=time.perf_counter(); model.fit(X_train,y_train); p=model.predict_proba(X_valid)[:,1]
    result={'model':name,'hypothesis':hypothesis,'configuration':configuration,'validation_start_key':int(valid_period['_arrival_key'].min()),'runtime_seconds':time.perf_counter()-started,**probability_metrics(y_valid,p)}
    experiments.append(result); models[name]=model; probabilities[name]=p
    return pd.DataFrame([result])
logistic=Pipeline([('prep',preprocess),('model',LogisticRegression(max_iter=2000,C=1.0,random_state=SEED))])
display(fit_and_record('Logistic',logistic,'C=1.0; max_iter=2000','A regularized linear baseline may generalize across the later period.'))

## 4. Random Forest benchmark

Carry forward the Week 6 bagging benchmark. Keep the rows, features, prediction moment, validation period, and metrics unchanged.

In [ ]:
forest=Pipeline([('prep',preprocess),('model',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,max_features='sqrt',n_jobs=1,random_state=SEED))])
display(fit_and_record('Random Forest',forest,'trees=300; min_leaf=20; max_features=sqrt','Averaging decorrelated trees may improve nonlinear ranking without fragile leaves.'))

## 5. Gradient boosting benchmark

This starter uses scikit-learn histogram gradient boosting so it runs without a separate package installation. It follows the Week 7 sequential loss-correction logic. A later declared experiment may use XGBoost or LightGBM while preserving the same evidence design and recording package versions and exact parameters.

The conservative configuration below is a starting point, not a preselected winner.

In [ ]:
boost_preprocess=ColumnTransformer([('num',numeric_pipe,numeric),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical)])
boosting=Pipeline([('prep',boost_preprocess),('model',HistGradientBoostingClassifier(learning_rate=0.05,max_iter=400,max_leaf_nodes=15,min_samples_leaf=30,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=30,random_state=SEED))])
display(fit_and_record('Histogram Gradient Boosting',boosting,'eta=.05; max_iter=400; leaves=15; min_leaf=30; L2=1; patience=30','Small sequential corrections may improve later-period probability quality.'))
print('Selected iterations:',boosting.named_steps['model'].n_iter_)
comparison=pd.DataFrame(experiments).sort_values('log_loss')
display(comparison[['model','configuration','runtime_seconds','log_loss','pr_auc','roc_auc','brier']])

### Notebook debrief

1. Which model has the strongest later-period log loss, and is the difference material?
2. Did PR-AUC, ROC-AUC, and Brier score tell a consistent story?
3. How many boosting iterations survived early stopping?
4. Which output needs a slice, calibration, or stability check?
5. State one hypothesis for the next experiment and change one choice only.

---

# Class hard stop

Stop here during the shared Week 7 run. Sections 6–9 are independent work. Do not use another person's final model, threshold, or recommendation.

## 6. Independent experiment record

Run at least two additional deliberate experiments so the completed table contains at least five runs. Record a hypothesis, one controlled change, the same validation period, runtime, metrics, best iteration when applicable, interpretation, and decision. Do not search many settings and report only the winner.

In [ ]:
# Add declared experiments with fit_and_record(...) or an equivalent function.
experiment_table=pd.DataFrame(experiments).copy()
experiment_table['interpretation']=''
experiment_table['decision']=''
display(experiment_table)

## 7. Calibration, deposit economics, and sensitivity

Use validation predictions only. Recalculate the threshold table after changing each supplied assumption. Keep model selection, probability evaluation, threshold selection, and operating policy distinct.

In [ ]:
COST=costs.set_index('parameter')['value'].to_dict()
room_revenue=valid_period['average_daily_rate'].clip(lower=0)*(valid_period['weekend_nights']+valid_period['weekday_nights']).clip(lower=1)
def policy_table(y,p,revenue,assumptions=COST,thresholds=np.arange(.10,.91,.05)):
    rows=[]
    for threshold in thresholds:
        deposit=p>=threshold
        recovered=(deposit*y*revenue*assumptions['empty_room_cost_rate']*assumptions['deposit_recovery_rate']).sum()
        friction=deposit.sum()*assumptions['deposit_friction_cost']
        rows.append({'threshold':threshold,'deposits':int(deposit.sum()),'deposits_per_1000':1000*deposit.mean(),'expected_value':recovered-friction,'false_positive_deposits':int((deposit & (np.asarray(y)==0)).sum())})
    return pd.DataFrame(rows)
candidate_name=comparison.iloc[0]['model']  # provisional for exploration only
candidate_policy=policy_table(y_valid.to_numpy(),probabilities[candidate_name],room_revenue.to_numpy())
display(candidate_name,candidate_policy.sort_values('expected_value',ascending=False).head())

In [ ]:
sensitivity_rows=[]
for parameter,multipliers in {'empty_room_cost_rate':[.75,1,1.25],'deposit_friction_cost':[.75,1,1.25],'deposit_recovery_rate':[.75,1,1.25]}.items():
    for multiplier in multipliers:
        scenario=COST.copy(); scenario[parameter]=COST[parameter]*multiplier
        tab=policy_table(y_valid.to_numpy(),probabilities[candidate_name],room_revenue.to_numpy(),scenario)
        best=tab.loc[tab.expected_value.idxmax()]
        sensitivity_rows.append({'parameter':parameter,'multiplier':multiplier,'threshold':best.threshold,'deposits_per_1000':best.deposits_per_1000,'expected_value':best.expected_value})
sensitivity=pd.DataFrame(sensitivity_rows)
display(sensitivity)

## 8. Error and policy analysis across operational groups

Analyze at least three pre-decision groupings. Report group size and prevalence alongside deposit rate, recall, false-positive burden, and calibration evidence. Use uncertainty language for small groups.

In [ ]:
def group_policy_view(frame,y,p,threshold,group):
    work=frame[[group]].copy(); work['y']=np.asarray(y); work['p']=np.asarray(p); work['deposit']=work.p>=threshold
    def summarize(g):
        pos=g.y.sum(); neg=len(g)-pos
        return pd.Series({'n':len(g),'prevalence':g.y.mean(),'mean_probability':g.p.mean(),'deposit_rate':g.deposit.mean(),'recall':((g.deposit)&(g.y==1)).sum()/pos if pos else np.nan,'false_positive_rate':((g.deposit)&(g.y==0)).sum()/neg if neg else np.nan})
    return work.groupby(group,dropna=False).apply(summarize,include_groups=False).sort_values('n',ascending=False)
example_threshold=float(candidate_policy.loc[candidate_policy.expected_value.idxmax(),'threshold'])
for group in ['hotel','market_segment','customer_type']:
    print('\n',group); display(group_policy_view(valid_period,y_valid,probabilities[candidate_name],example_threshold,group))

## 9. Final model, threshold, memo, and decision file

Record before generating test predictions:

1. Final model and exact configuration: **[write here]**
2. Evidence that earned selection: **[write here]**
3. Final threshold and deposits per 1,000: **[write here]**
4. Expected value and sensitivity conclusion: **[write here]**
5. Important group failure or guest-experience concern: **[write here]**
6. One condition under which the policy should not be used: **[write here]**

In [ ]:
FINAL_MODEL_NAME=None
FINAL_THRESHOLD=None
if FINAL_MODEL_NAME is None or FINAL_THRESHOLD is None:
    raise ValueError('Record FINAL_MODEL_NAME and FINAL_THRESHOLD after completing validation evidence.')
final_model=models[FINAL_MODEL_NAME]
final_model.fit(train[FEATURES],train[TARGET])
test_probability=final_model.predict_proba(test[FEATURES])[:,1]
submission=pd.DataFrame({'booking_id':test[ID],'cancellation_probability':test_probability,'require_deposit':(test_probability>=FINAL_THRESHOLD).astype(int)})
submission.to_csv('submission.csv',index=False)
display(submission.head(),submission.require_deposit.value_counts(normalize=True).rename('share'))

### Submission check

- Restart and run every completed cell from top to bottom.
- Preserve at least five deliberate experiments.
- Validate `submission.csv` with `checks/validate_submission.py`.
- Inspect the file with the shared output explorer.
- Submit the executed notebook, valid CSV, one-page memo, and AI-use disclosure through Blackboard.
- Blackboard is the source of truth for the due date and submission field.

## Generative AI use disclosure

Complete `../../shared/ai_use_disclosure.md`. Record the tool, task, affected material, important error or unsafe assumption, correction, and verification. You remain responsible for every submitted decision and result.